# Notebook 01 — Data Loading and Pipeline Sanity Checks

**Goal:** Verify the three M5 raw files load cleanly, run the end-to-end pipeline in `src/data.py`, and confirm the resulting weekly panel has sensible shape, coverage, and statistics before we invest time in EDA.

**Pipeline summary:**
1. Load `calendar.csv` (dates, events, SNAP)
2. Load wide `sales_train_evaluation.csv` and filter to items with non-zero sales before 2013-01-01 (full-history items)
3. Build per-(item, store, week) discount-depth features from `sell_prices.csv`
4. For each of the 10 stores: melt → aggregate → join prices → aggregate to dept-week
5. Concat all stores; add event/SNAP features
6. Cache as parquet at `data/interim/m5_weekly_panel.parquet`

**Downstream:** Notebook 02 (EDA) and 03 (baseline model) both load the cached parquet, not the raw files.

## Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src import data as m5_data

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print(f"pandas {pd.__version__}, numpy {np.__version__}")
print(f"repo root: {REPO_ROOT}")

pandas 2.3.3, numpy 2.4.4
repo root: /Users/avima01/ucb-cap-op-promo-str


## 1. Sanity-check raw files

Quick checks on each raw file: shape, head, dtypes, date range. We're looking for surprises before we trust the pipeline.

### 1a. `calendar.csv`

In [2]:
cal = m5_data.load_calendar()
print(f"shape: {cal.shape}")
print(f"date range: {cal['date'].min().date()} → {cal['date'].max().date()}")
print(f"events flagged on {cal['has_event'].sum()} days")
print(f"SNAP days — CA: {cal['snap_CA'].sum()}, TX: {cal['snap_TX'].sum()}, WI: {cal['snap_WI'].sum()}")
cal.head()

shape: (1969, 16)
date range: 2011-01-29 → 2016-06-19
events flagged on 162 days
SNAP days — CA: 650, TX: 650, WI: 650


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,week_start,has_event
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,2011-01-25,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0,2011-01-25,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0,2011-01-25,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0,2011-02-01,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1,2011-02-01,0


### 1b. `sales_train_evaluation.csv` (wide format)

Inspecting the wide file directly. The `d_1, d_2, ... d_1941` columns each hold one day's unit sales for one (item, store) pair.

In [3]:
sales_wide_peek = pd.read_csv(m5_data.M5_DIR / "sales_train_evaluation.csv", nrows=5)
print(f"file columns: {sales_wide_peek.shape[1]} (first 6 are id cols, rest are day cols)")
print(f"id columns: {[c for c in sales_wide_peek.columns if not c.startswith('d_')]}")
print(f"day columns: {sales_wide_peek.columns[6]} ... {sales_wide_peek.columns[-1]}")

# Quick view of the first 5 rows × first 12 cols
sales_wide_peek.iloc[:, :12]

file columns: 1947 (first 6 are id cols, rest are day cols)
id columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day columns: d_1 ... d_1941


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0


In [4]:
# Item/store counts from the full file (lightweight read of id columns only)
id_only = pd.read_csv(
    m5_data.M5_DIR / "sales_train_evaluation.csv",
    usecols=["item_id", "dept_id", "cat_id", "store_id", "state_id"],
)
print(f"total rows (item × store pairs): {len(id_only):,}")
print(f"unique items: {id_only['item_id'].nunique():,}")
print(f"unique stores: {id_only['store_id'].nunique()}")
print(f"unique departments: {id_only['dept_id'].nunique()}")
print()
print("stores per state:")
print(id_only.groupby('state_id')['store_id'].nunique())
print()
print("items per category:")
print(id_only.groupby('cat_id')['item_id'].nunique())
print()
print("items per department:")
print(id_only.groupby('dept_id')['item_id'].nunique())

total rows (item × store pairs): 30,490
unique items: 3,049
unique stores: 10
unique departments: 7

stores per state:
state_id
CA    4
TX    3
WI    3
Name: store_id, dtype: int64

items per category:
cat_id
FOODS        1437
HOBBIES       565
HOUSEHOLD    1047
Name: item_id, dtype: int64

items per department:
dept_id
FOODS_1        216
FOODS_2        398
FOODS_3        823
HOBBIES_1      416
HOBBIES_2      149
HOUSEHOLD_1    532
HOUSEHOLD_2    515
Name: item_id, dtype: int64


### 1c. `sell_prices.csv`

Weekly average sell price per (item, store, fiscal week). One row per priced week — if an item isn't in a store's assortment that week, no row.

In [5]:
prices_peek = pd.read_csv(m5_data.M5_DIR / "sell_prices.csv")
print(f"shape: {prices_peek.shape}")
print(f"unique items priced: {prices_peek['item_id'].nunique():,}")
print(f"unique stores: {prices_peek['store_id'].nunique()}")
print(f"unique weeks (wm_yr_wk): {prices_peek['wm_yr_wk'].nunique()}")
print(f"price range: ${prices_peek['sell_price'].min():.2f} → ${prices_peek['sell_price'].max():.2f}")
print(f"median price: ${prices_peek['sell_price'].median():.2f}")
prices_peek.head()

shape: (6841121, 4)
unique items priced: 3,049


unique stores: 10
unique weeks (wm_yr_wk): 282
price range: $0.01 → $107.32
median price: $3.47


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


## 2. Run the end-to-end pipeline

Calls `build_weekly_panel()` which:
- loads calendar
- filters wide sales to full-history items (those with non-zero sales before 2013-01-01)
- builds discount-depth features (rolling-median operational definition)
- per-store: melts → item-week aggregates → joins prices → dept-week aggregates
- concatenates, adds calendar/SNAP features, caches parquet

**Expected runtime:** ~1-3 minutes on a modern laptop. The per-store loop is the heavy step.

In [6]:
import time
t0 = time.time()
weekly = m5_data.build_weekly_panel()
print(f"\npipeline complete in {time.time() - t0:.1f}s")

[1/5] calendar loaded: 1,969 rows, 2011-01-29 → 2016-06-19


[2/5] sales wide loaded + filtered: 23,070 (item, store) rows × 1941 days
      kept items with non-zero sales before 2013-01-01


[3/5] price features built: 13,651,752 (item, store, week) rows, 0.4% flagged is_promotion


[4/5]   store 1/10 (CA_1): 1,946 dept-weeks


[4/5]   store 2/10 (CA_2): 1,946 dept-weeks


...

[4/5]   store 6/10 (TX_2): 1,946 dept-weeks


[4/5]   store 7/10 (TX_3): 1,946 dept-weeks


[4/5]   store 8/10 (WI_1): 1,946 dept-weeks


[4/5]   store 9/10 (WI_2): 1,946 dept-weeks


[4/5]   store 10/10 (WI_3): 1,946 dept-weeks
[5/5] final panel: 19,460 rows, 2011-01-25 → 2016-05-17
      cached to data/interim/m5_weekly_panel.parquet

pipeline complete in 28.6s


## 3. Inspect the weekly panel

In [7]:
print(f"shape: {weekly.shape}")
print(f"date range: {weekly['week_start'].min().date()} → {weekly['week_start'].max().date()}")
print(f"unique weeks: {weekly['week_start'].nunique()}")
print(f"unique (dept × store) combos: {weekly.groupby(['dept_id', 'store_id']).ngroups}")
print()
weekly.head()

shape: (19460, 15)
date range: 2011-01-25 → 2016-05-17
unique weeks: 278
unique (dept × store) combos: 70



/var/folders/x0/nn3ptf7n1gj6vsmwltc3nbmsm2pmfv/T/ipykernel_6853/1230450742.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(f"unique (dept × store) combos: {weekly.groupby(['dept_id', 'store_id']).ngroups}")


,dept_id,cat_id,store_id,state_id,week_start,unit_sales,revenue,n_items_priced,n_items_on_promo,promo_share,mean_discount_depth,mean_sell_price,sales_weighted_discount_depth,has_event_days,snap_days_in_week
0,FOODS_1,FOODS,CA_1,CA,2011-01-25,795.0,1776.010010,101,0.0,0.0,0.0,2.989109,0.0,0,0
1,FOODS_1,FOODS,CA_1,CA,2011-02-01,2901.0,6764.569824,107,0.0,0.0,0.0,3.003269,0.0,1,7
2,FOODS_1,FOODS,CA_1,CA,2011-02-08,4132.0,10342.740234,110,0.0,0.0,0.0,3.007926,0.0,1,3
3,FOODS_1,FOODS,CA_1,CA,2011-02-15,3116.0,7032.640137,112,0.0,0.0,0.0,2.988198,0.0,1,0
4,FOODS_1,FOODS,CA_1,CA,2011-02-22,3030.0,7114.740234,113,0.0,0.0,0.0,2.972800,0.0,0,0


In [8]:
weekly.dtypes

dept_id                                category
cat_id                                 category
store_id                               category
state_id                               category
week_start                       datetime64[ns]
unit_sales                              float32
revenue                                 float32
n_items_priced                            int64
n_items_on_promo                        float64
promo_share                             float64
mean_discount_depth                     float32
mean_sell_price                         float32
sales_weighted_discount_depth           float32
has_event_days                             int8
snap_days_in_week                          int8
dtype: object

In [9]:
weekly.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
dept_id,19460,7,FOODS_1,2780,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cat_id,19460,3,FOODS,8340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_id,19460,10,CA_1,1946,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state_id,19460,3,CA,7784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
week_start,19460,NaN,NaN,NaN,2013-09-20 12:00:00,2011-01-25 00:00:00,2012-05-22 00:00:00,2013-09-20 12:00:00,2015-01-20 00:00:00,2016-05-17 00:00:00,NaN
unit_sales,19460.0,NaN,NaN,NaN,6209.183105,6.0,1725.0,3729.0,6927.5,54803.0,7408.841797
revenue,19460.0,NaN,NaN,NaN,17386.537109,16.619999,6777.990112,12744.639648,23082.537598,113423.523438,16272.975586
n_items_priced,19460.0,NaN,NaN,NaN,298.007862,12.0,188.0,296.0,389.0,596.0,147.155239
n_items_on_promo,19460.0,NaN,NaN,NaN,2.76593,0.0,0.0,1.0,3.0,63.0,4.982774
promo_share,19460.0,NaN,NaN,NaN,0.004464,0.0,0.0,0.001529,0.005319,0.176056,0.008523


## 4. Sanity checks on the panel

In [10]:
# Coverage: do we have all (dept, store) pairs across most weeks?
coverage = (
    weekly.groupby(["dept_id", "store_id"], observed=True)
    .size()
    .reset_index(name="n_weeks")
)
print(f"weeks per (dept, store) — min/median/max: "
      f"{coverage['n_weeks'].min()} / {coverage['n_weeks'].median():.0f} / {coverage['n_weeks'].max()}")
print(f"# of (dept, store) pairs with < 200 weeks: {(coverage['n_weeks'] < 200).sum()}")

weeks per (dept, store) — min/median/max: 278 / 278 / 278
# of (dept, store) pairs with < 200 weeks: 0


In [11]:
# Promotion prevalence
any_promo = (weekly["promo_share"] > 0).mean() * 100
med_share = weekly["promo_share"].median() * 100
p95_share = weekly["promo_share"].quantile(0.95) * 100
print(f"% of (dept, store, week) cells with ANY promotion:        {any_promo:.1f}%")
print(f"median promo_share across all cells:                       {med_share:.2f}%")
print(f"95th percentile promo_share:                               {p95_share:.2f}%")

non_zero_depth = weekly[weekly["sales_weighted_discount_depth"] > 0]["sales_weighted_discount_depth"]
if len(non_zero_depth) > 0:
    print(f"\ndiscount depth (where > 0) — quantiles:")
    print((non_zero_depth.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95]) * 100).round(2).to_string())

% of (dept, store, week) cells with ANY promotion:        57.3%
median promo_share across all cells:                       0.15%
95th percentile promo_share:                               1.86%

discount depth (where > 0) — quantiles:
0.10    0.00
0.25    0.01
0.50    0.04
0.75    0.13
0.90    0.34
0.95    0.62


In [12]:
# Per-department snapshot
dept_snapshot = (
    weekly.groupby("dept_id", observed=True)
    .agg(
        n_weeks=("week_start", "nunique"),
        n_stores=("store_id", "nunique"),
        mean_weekly_units=("unit_sales", "mean"),
        mean_promo_share=("promo_share", "mean"),
        mean_disc_depth=("sales_weighted_discount_depth", "mean"),
    )
    .round(3)
)
dept_snapshot

,n_weeks,n_stores,mean_weekly_units,mean_promo_share,mean_disc_depth
dept_id,,,,,
FOODS_1,278,10,3441.904053,0.005,0.001
FOODS_2,278,10,5115.062988,0.008,0.001
FOODS_3,278,10,21416.128906,0.007,0.002
HOBBIES_1,278,10,3712.121094,0.003,0.001
HOBBIES_2,278,10,321.332001,0.003,0.001
HOUSEHOLD_1,278,10,7555.957031,0.003,0.001
HOUSEHOLD_2,278,10,1901.774048,0.003,0.001


In [13]:
# Per-state snapshot
state_snapshot = (
    weekly.groupby("state_id", observed=True)
    .agg(
        n_stores=("store_id", "nunique"),
        mean_weekly_units=("unit_sales", "mean"),
        mean_promo_share=("promo_share", "mean"),
        mean_snap_days=("snap_days_in_week", "mean"),
    )
    .round(3)
)
state_snapshot

,n_stores,mean_weekly_units,mean_promo_share,mean_snap_days
state_id,,,,
CA,4,6754.101074,0.004,2.302
TX,3,5996.367188,0.006,2.302
WI,3,5695.440918,0.004,2.302


In [14]:
# Missing values check — should be zero or near-zero in the final panel
missing = weekly.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "  none")

Missing values per column:
  none


## 5. Summary

Key outputs for downstream notebooks:

- Weekly panel cached at `data/interim/m5_weekly_panel.parquet`.
- Grain: `(dept_id × store_id × week_start)`.
- Columns: identifiers + `unit_sales`, `revenue`, `n_items_priced`, `n_items_on_promo`, `promo_share`, `mean_discount_depth`, `sales_weighted_discount_depth`, `mean_sell_price`, `has_event_days`, `snap_days_in_week`.

Notebooks 02 (EDA) and 03 (baseline model) reload via `m5_data.load_weekly_panel()` — should be near-instant from the parquet cache.